# Custom hybrid retriever — end-to-end smoke test

Exercises `searcher.searchers.HybridSearcher` (BM25 + FAISS dense retrieval, fused via
RRF, reranked with Qwen3-Reranker) first on its own, then behind the same tool-calling
loop the search agent CLIs use — the class wired up as `--searcher-type custom`
(see [docs/custom_retriever.md](../docs/custom_retriever.md)).

Uses the smaller `Qwen3-Embedding-0.6B` index instead of the 8B one so this can run
alongside other GPU users on this box without contending for VRAM.

**Needs:** `indexes/bm25` and `indexes/qwen3-embedding-0.6b` built, plus an
OpenAI-compatible generation endpoint (section 2).

1. Setup
2. Build the hybrid searcher — search, `get_document`, shape checks
3. Generation endpoint (vLLM)
4. End-to-end: vLLM generation + `HybridSearcher` retrieval
5. Full run over all queries (CLI)
6. Evaluate a run
    - Appendix: Azure OpenAI as the generation backend

## 1. Setup

The notebook lives in `notebooks/`, but every path below is repo-relative — same as the
CLIs — so resolve the repo root once and work from there.

In [ ]:
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Nearest ancestor holding pyproject.toml, so this runs from any working dir."""
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError(f"No pyproject.toml found above {start}")


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)

for path in (REPO_ROOT, REPO_ROOT / "search_agent"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print("repo root:", REPO_ROOT)

## 2. Build the hybrid searcher

Args come from `HybridSearcher.parse_args`, the same parser the CLIs use, so every
default (pooling, dtype, `retrieve_k`/`rrf_k`, reranker model, ...) stays in sync with
the class instead of being duplicated here. Only the overrides are spelled out.

In [ ]:
import argparse

from searcher.searchers import HybridSearcher

parser = argparse.ArgumentParser()
HybridSearcher.parse_args(parser)

args = parser.parse_args(
    [
        "--bm25-index-path", "indexes/bm25",
        "--faiss-index-path", "indexes/qwen3-embedding-0.6b/corpus.shard*_of_4.pkl",
        "--embedding-model-name", "Qwen/Qwen3-Embedding-0.6B",
    ]
)

searcher = HybridSearcher(args)
print("search_type:", searcher.search_type)

In [ ]:
query = "What country has been culture and history are the most important in west world (france)"
k = 5

results = searcher.search(query, k=k)

print(f"Query: {query!r}  (top-{k}, hybrid + rerank)\n")
for i, hit in enumerate(results, 1):
    snippet = hit["text"][:200].replace("\n", " ")
    print(f"{i}. docid={hit['docid']}  score={hit['score']:.4f}")
    print(f"   {snippet}...\n")

In [ ]:
# get_document should resolve any docid returned above to its full text
doc = searcher.get_document(results[0]["docid"])
assert doc is not None
print("docid:", doc["docid"])
print(doc["text"][:500])

# unknown docid should come back None, not raise
assert searcher.get_document("not-a-real-docid") is None
print("\nget_document(unknown) -> None  OK")

In [ ]:
# Sanity checks on shape/ordering
assert len(results) == k
assert all({"docid", "score", "text"} <= hit.keys() for hit in results)
assert results == sorted(results, key=lambda h: h["score"], reverse=True), "results should be sorted by score desc"
assert len({hit["docid"] for hit in results}) == len(results), "no duplicate docids"

print("search_description:", searcher.search_description(k))
print("get_document_description:", searcher.get_document_description())
print("\nAll shape/ordering checks passed.")

## 3. Generation endpoint (vLLM)

External OpenAI-compatible server, configured entirely from the environment
(`VLLM_BASE_URL`, `VLLM_API_KEY`, `VLLM_MODEL`) — set these in the repo-root `.env`
rather than hardcoding an endpoint here. The defaults below assume a local server.

In [ ]:
from openai import OpenAI

VLLM_BASE_URL = os.environ.get("VLLM_BASE_URL", "http://localhost:8000/v1")
VLLM_API_KEY = os.environ.get("VLLM_API_KEY", "EMPTY")
MODEL = os.environ.get("VLLM_MODEL", "Qwen/Qwen3.5-9B")

client = OpenAI(base_url=VLLM_BASE_URL, api_key=VLLM_API_KEY)
print("served models:", [m.id for m in client.models.list().data])

In [ ]:
# Connectivity smoke test
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with the single word: pong"}],
    max_tokens=16,
)
print(response.choices[0].message.content)

## 4. End-to-end: vLLM generation + `HybridSearcher` retrieval

Reuses the tool-calling loop `search_agent/oss_client.py` uses for full runs
(`SearchToolHandler` + `run_conversation_with_tools`, via the Responses API), wired to
the endpoint from section 3 and the `searcher` built in section 2 rather than a
CLI-constructed one.

In [ ]:
from oss_client import SearchToolHandler, run_conversation_with_tools
from prompts import format_query

tool_handler = SearchToolHandler(
    searcher=searcher, snippet_max_tokens=512, k=5, include_get_document=True
)

question = "What country has been culture and history are the most important in west world (france)"

initial_request = {
    "model": MODEL,
    "max_output_tokens": 4000,
    "input": [
        {"role": "user", "content": format_query(question, "QUERY_TEMPLATE_NO_GET_DOCUMENT")}
    ],
    "tools": tool_handler.get_tool_definitions(),
    "truncation": "auto",
    "reasoning": {"effort": "medium", "summary": "detailed"},
}

final_messages, tool_usage, status = run_conversation_with_tools(
    client, initial_request, tool_handler, max_iterations=10, verbose=True
)

print("status:", status)
print("tool_usage:", tool_usage)

In [ ]:
for m in final_messages:
    if m.get("type") == "message":
        for part in m.get("content", []) or []:
            if part.get("type") == "output_text":
                print(part["text"])

## 5. Full run over all queries (CLI)

The same wiring, over `topics-qrels/queries.tsv` instead of one question — run from the
repo root, with the larger 8B index since it is not sharing the GPU with a notebook
kernel. Writes one JSON transcript per query under `--output-dir`.

```bash
uv run python search_agent/oss_client.py \
  --model "Qwen/Qwen3.5-9B" \
  --model-url "$VLLM_BASE_URL" \
  --api-key "$VLLM_API_KEY" \
  --searcher-type custom \
  --bm25-index-path indexes/bm25 \
  --faiss-index-path "indexes/qwen3-embedding-8b/corpus.shard*_of_4.pkl" \
  --embedding-model-name Qwen/Qwen3-Embedding-8B \
  --output-dir runs/custom/qwen3.5-9b \
  --query topics-qrels/queries.tsv \
  --num-threads 10 \
  --get-document
```

## 6. Evaluate a run

LLM-as-judge over the transcripts written above (see
[docs/llm_as_judge.md](../docs/llm_as_judge.md)); results land in `evals/<run>/`.

In [ ]:
!uv run python scripts_evaluation/evaluate_with_openai.py --input_dir runs/custom/qwen3.5-9b

Recorded result for `runs/custom/qwen3.5-9b` (830 evaluated, 2026-08-01):

| metric | value |
| --- | --- |
| Accuracy | 57.35% |
| Recall | 63.33% |
| Calibration error | 28.70% |
| Avg tool calls | search 14.68, get_document 1.70 |
| Responses with citations | 750/830 (90.36%) |
| Citation precision / recall (avg) | 69.43% / 49.25% |

### Appendix: Azure OpenAI as the generation backend

Alternative to the vLLM endpoint in section 3; credentials come from the repo-root
`.env`. Kept as `azure_client` so it does not clobber the `client` the agent loop uses —
to drive section 4 with it, pass `azure_client` to `run_conversation_with_tools` and set
`MODEL` to the deployment name (needs an `AZURE_OPENAI_API_VERSION` that supports the
Responses API).

In [ ]:
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv(REPO_ROOT / ".env")

azure_client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
)

response = azure_client.chat.completions.create(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    messages=[{"role": "user", "content": "Reply with the single word: pong"}],
    max_tokens=16,
)
print(response.choices[0].message.content)